# Graph-DTW results tables (OSM ↔ Sweden NVDB)

Run the full route-based pipeline and show the two output tables:
- **routes_summary** — one row per OSM A-edge (the chosen route + whole-edge metrics).
- **routes_long** — one row per (A-edge, B-edge in its route): the result **divided per B-edge**,
  with `seq` (order of matching), coverage of A, % of the B-edge used, and per-edge bearing Δ.


In [1]:
import sys, os, time
import pandas as pd
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
sys.path.append(os.path.abspath(".."))
from network_matching import DuckDBMapMatcher, setup_logging

setup_logging(log_dir="../logs", console=False)

# One-call initializer: load both WKT-CSV networks + configure + set search radius.
m = DuckDBMapMatcher.from_wkt_csv(
    "../data/osm_edges.csv", "../data/sweden_edges.csv",
    id_a="edge_id", id_b="directed_id", utm_srid=3006, max_distance=30)

t = time.time()
routes_long, routes_summary = m.match_routes(snap_tolerance_m=0.5, step_meters=10, trim_ends_m=1.0, n_jobs=-1)
print(f"match_routes done in {time.time()-t:.1f}s")
print(f"routes_summary: {routes_summary.shape}   routes_long: {routes_long.shape}")
print(routes_summary['match_type'].value_counts())


match_routes done in 6.6s
routes_summary: (3948, 11)   routes_long: (5423, 16)
match_type
1:1          2079
1:N_ROUTE    1329
NO_MATCH      540
Name: count, dtype: int64


## routes_summary — one row per OSM edge

In [2]:
matched = routes_summary[routes_summary.match_type != 'NO_MATCH']
cols = ['source_id','n_edges','dest_ids','dtw_distance','max_dtw_distance','overlap_pct','matched_len','bearing_diff','match_type']
print(matched.sort_values('n_edges', ascending=False)[cols].head(10).to_string(index=False))
display(matched.sort_values('n_edges', ascending=False)[cols].head(10))


 source_id  n_edges                          dest_ids  dtw_distance  max_dtw_distance overlap_pct  matched_len  bearing_diff match_type
      2137        6 [676, 549, 1821, 1820, 1654, 808]      3.959152          6.880171         100   499.194565      0.285697  1:N_ROUTE
      1256        6    [542, 540, 539, 537, 535, 536]      2.944632          5.161546         100   298.469968      0.162042  1:N_ROUTE
       930        6    [51, 1398, 1399, 26, 1167, 23]     16.499279         50.281391         100   240.692380      7.880304  1:N_ROUTE
      1560        5    [1229, 1233, 1230, 1228, 1106]      1.001944          2.840327         100   167.367525      0.287593  1:N_ROUTE
      2175        5       [1668, 1120, 323, 324, 325]      2.308840          8.512822         100    64.403722      1.019552  1:N_ROUTE
      3645        5    [1306, 1307, 1308, 1334, 1341]      0.841821          2.302012         100   346.449370      0.264165  1:N_ROUTE
      3556        5        [927, 849, 941, 930, 

,source_id,n_edges,dest_ids,dtw_distance,max_dtw_distance,overlap_pct,matched_len,bearing_diff,match_type
1872,2137,6,"[676, 549, 1821, 1820, 1654, 808]",3.959152,6.880171,100,499.194565,0.285697,1:N_ROUTE
1103,1256,6,"[542, 540, 539, 537, 535, 536]",2.944632,5.161546,100,298.469968,0.162042,1:N_ROUTE
823,930,6,"[51, 1398, 1399, 26, 1167, 23]",16.499279,50.281391,100,240.692380,7.880304,1:N_ROUTE
1363,1560,5,"[1229, 1233, 1230, 1228, 1106]",1.001944,2.840327,100,167.367525,0.287593,1:N_ROUTE
1904,2175,5,"[1668, 1120, 323, 324, 325]",2.308840,8.512822,100,64.403722,1.019552,1:N_ROUTE
3142,3645,5,"[1306, 1307, 1308, 1334, 1341]",0.841821,2.302012,100,346.449370,0.264165,1:N_ROUTE
3063,3556,5,"[927, 849, 941, 930, 2131]",1.600294,4.970016,100,260.122963,0.112271,1:N_ROUTE
25,26,5,"[1533, 1534, 169, 1536, 1851]",1.451992,4.136213,100,99.507963,0.552128,1:N_ROUTE
1313,1508,5,"[409, 380, 381, 382, 383]",1.322146,4.670186,100,328.761150,0.070771,1:N_ROUTE
1320,1515,5,"[1182, 1422, 37, 35, 1406]",1.571981,5.744032,100,636.786891,0.051883,1:N_ROUTE


## routes_long — per B-edge in each route (the result divided per edge)

`seq` = order of matching; `edge_cover_pct` = % of OSM edge covered; `edge_b_used_pct` = % of that
NVDB edge used; `edge_bearing_diff` = bearing diff for that segment.

In [3]:
long_cols = ['source_id','dest_id','seq','direction','edge_match_dist_avg','edge_a_len',
             'edge_cover_pct','edge_matched_len','edge_b_len','edge_b_used_pct','edge_bearing_diff','n_points']
print(routes_long[long_cols].head(12).to_string(index=False))
display(routes_long[long_cols].head(12))


 source_id  dest_id  seq direction  edge_match_dist_avg  edge_a_len  edge_cover_pct  edge_matched_len  edge_b_len  edge_b_used_pct  edge_bearing_diff  n_points
         1    580.0    0   forward             2.059110   43.288801           100.0         47.666485   58.998596             80.8           2.058297        12
         2    580.0    0   forward             2.044135    4.253289             6.9          4.264913   58.998596              7.2           0.325796         2
         2    589.0    1   forward             2.866438   57.766224            93.1         58.691019   59.234199             99.1           3.514301        14
         3    339.0    0   forward             0.477514   72.775683            31.0         72.777376  734.789390              9.9           0.355603        10
         3    411.0    1   forward             3.614639   23.263275             9.9         23.265065   23.265065            100.0           0.710869         3
         3    345.0    2   forward      

,source_id,dest_id,seq,direction,edge_match_dist_avg,edge_a_len,edge_cover_pct,edge_matched_len,edge_b_len,edge_b_used_pct,edge_bearing_diff,n_points
0,1,580.0,0,forward,2.059110,43.288801,100.0,47.666485,58.998596,80.8,2.058297,12
1,2,580.0,0,forward,2.044135,4.253289,6.9,4.264913,58.998596,7.2,0.325796,2
2,2,589.0,1,forward,2.866438,57.766224,93.1,58.691019,59.234199,99.1,3.514301,14
3,3,339.0,0,forward,0.477514,72.775683,31.0,72.777376,734.789390,9.9,0.355603,10
4,3,411.0,1,forward,3.614639,23.263275,9.9,23.265065,23.265065,100.0,0.710869,3
5,3,345.0,2,forward,3.439839,138.685777,59.1,139.733789,160.666417,87.0,2.799321,26
6,4,1659.0,0,forward,24.904947,146.352031,100.0,85.090419,131.056021,64.9,14.855059,19
7,5,2319.0,0,forward,2.133325,231.530909,100.0,231.026357,278.256747,83.0,1.614251,33
8,6,476.0,0,forward,3.121705,119.397511,94.0,119.388305,160.899316,74.2,0.589361,15
9,6,1771.0,1,backward,2.360174,7.654512,6.0,7.676594,252.879321,3.0,0.000000,1


## One edge, full per-edge breakdown (OSM 3597)

In [4]:
one = routes_long[routes_long.source_id == 3597][long_cols].sort_values('seq')
print(one.to_string(index=False))
display(one)


 source_id  dest_id  seq direction  edge_match_dist_avg  edge_a_len  edge_cover_pct  edge_matched_len  edge_b_len  edge_b_used_pct  edge_bearing_diff  n_points
      3597    307.0    0  backward             0.224936   18.104738             4.9         17.471677   24.850772             70.3           0.171819         6
      3597   1662.0    1   forward             1.707838  347.739749            94.6        347.802852  347.802852            100.0           0.890452        47
      3597   1151.0    2  backward             2.428424    1.840743             0.5          2.049395    5.681670             36.1           0.000000         1


,source_id,dest_id,seq,direction,edge_match_dist_avg,edge_a_len,edge_cover_pct,edge_matched_len,edge_b_len,edge_b_used_pct,edge_bearing_diff,n_points
4970,3597,307.0,0,backward,0.224936,18.104738,4.9,17.471677,24.850772,70.3,0.171819,6
4971,3597,1662.0,1,forward,1.707838,347.739749,94.6,347.802852,347.802852,100.0,0.890452,47
4972,3597,1151.0,2,backward,2.428424,1.840743,0.5,2.049395,5.681670,36.1,0.000000,1


## Save the tables to `output/`

In [5]:
import os
os.makedirs("../output", exist_ok=True)
routes_summary.to_csv("../output/routes_summary.csv", index=False)
routes_long.to_csv("../output/routes_long.csv", index=False)
print("saved:")
print("  ../output/routes_summary.csv ", routes_summary.shape)
print("  ../output/routes_long.csv    ", routes_long.shape)


saved:
  ../output/routes_summary.csv  (3948, 11)
  ../output/routes_long.csv     (5423, 16)
